In [ ]:
import stim
import logging

logging.basicConfig(level=logging.ERROR)
from library.circuitry import Circuitry
from library.common import Pauli
from library.qubit_array import QubitArray
from library.steane_code.patch import SteaneCodePatch
from utils.simulation.clifft import sample

In [ ]:
def detector_report(circuitry: stim.Circuit) -> str:
    issues = []
    if len(circuitry.missing_detectors()) > 0:
        issues.append("MISSING")

    try:
        circuitry.detector_error_model(allow_gauge_detectors=False)
    except ValueError:
        issues.append("GAUGE")

    return "&".join(issues)

In [ ]:
def append_cultivation(steane: SteaneCodePatch, circuitry: Circuitry):
    all_qubits = steane.qubits
    meas_qubits = [ steane.qubits[q] for q in [ 10, 13, 14, 15] ]
    circuitry.append(f"{steane.injection.name}_DAG", all_qubits[1])
    circuitry.append("H", [ all_qubits[0], all_qubits[3] ])
    circuitry.append("RX", meas_qubits)
    circuitry.append_tick()

    # Do the thing.
    circuitry.append("CX", [ all_qubits[q] for q in [ 10, 5, 13, 1, 15, 3 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 13, 0, 15 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 10, 14, 0 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 14, 6 ]])
    circuitry.append_tick()
    circuitry.append("MX", meas_qubits[2])
    qubits.record_measurement(meas_qubits[2], f"CULT:X0")
    circuitry.append_tick()
    circuitry.append("RX", meas_qubits[2])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 14, 6 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 10, 14, 0 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 6, 13, 0, 15 ]])
    circuitry.append_tick()
    circuitry.append("CX", [ all_qubits[q] for q in [ 10, 5, 13, 1, 15, 3 ]])
    circuitry.append_tick()

    circuitry.append("MX", meas_qubits)
    measured_ancilla = [ all_qubits[q] for q in [10, 11, 13, 14, 15] ]
    for index, xa in enumerate(measured_ancilla):
        if index == 1:
            continue
        qubits.record_measurement(xa, f"CULT:X{index+1}")

    circuitry.append("H", [ all_qubits[0], all_qubits[3] ])
    circuitry.append(steane.injection.name, all_qubits[1])
    circuitry.append_tick()

In [ ]:
FILEROOT = "../generated/cultivation-stage-layout1"
scenarios : dict[str, Circuitry] = {}
observables = [ Pauli.X, Pauli.Y, Pauli.Z ]

In [ ]:
for observable in observables:
    qubits = QubitArray(dimensions=(5,3))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(qubits, injection=SteaneCodePatch.Injection.T)

    # Generate the circuit up to and including cultivation.
    steane.append_preparation(circuitry)
    for rnd in range(3):
        steane.append_superdense_cycle(circuitry, prefix=f"SDC{rnd}")
    # Append the experimental weight-5 cultivation.
    append_cultivation(steane, circuitry)
    steane.annotate_detectors(circuitry, sdc_rounds=3)

    circuitry.append_observable(
        0, f"CULTIVATION", steane.logical(observable) if observable != Pauli.Y else {
            steane.qubits[5] : "X", steane.qubits[6] : "X", steane.qubits[1] : "Y", steane.qubits[0] : "Z" , steane.qubits[3] : "Z"
        }, flip=observable == Pauli.Y
    )

    scenario = rf'Weight-5 [$\overline{{\mathbf{{{observable.name}}}}}$]'
    scenarios[scenario] = circuitry
    circuitry.to_file(FILEROOT + f".weight5.cultivation-t.{observable.name.lower()}-basis")

In [ ]:
for observable in observables:
    qubits = QubitArray(dimensions=(5,3))
    circuitry = Circuitry(qubits, clifford=False)
    steane = SteaneCodePatch(qubits, injection=SteaneCodePatch.Injection.T)

    # Generate the circuit up to and including cultivation.
    steane.append_preparation(circuitry)
    for rnd in range(3):
        steane.append_superdense_cycle(circuitry, prefix=f"SDC{rnd}")
    # Append the usual weight-7 cultivation.
    steane.append_cultivation(circuitry)
    steane.annotate_detectors(circuitry, sdc_rounds=3)

    circuitry.append_observable(0, f"CULTIVATION", steane.logical(observable))

    scenario = rf'Weight-7 [$\overline{{\mathbf{{{observable.name}}}}}$]'
    scenarios[scenario] = circuitry
    circuitry.to_file(FILEROOT + f".weight7.cultivation-t.{observable.name.lower()}-basis")

In [ ]:
sample(scenarios, title=r"Cultivation stage for $|\mathbf{T}\rangle$", label="Circuitry", fontsize=10)